In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(hooke)
    library(forcats)
    library(stringr)
    library(patchwork)
    library(cowplot)
    library(lisi)
})

In [ ]:
cds_VCS = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered_final.RDS")


In [ ]:
metadata = as.data.frame(colData(cds_VCS))

In [ ]:
metadata %>%
    group_by(Marker) %>% count()

metadata %>%
    distinct(Experiment, Embryo, Marker) %>%
    group_by(Marker) %>% count()

median(metadata$RNA_UMIs)
dim(metadata)

In [ ]:
## Supplementary Figure 6A

options(repr.plot.width=9, repr.plot.height=4, repr.plot.res=300)
metadata %>%
    group_by(Marker, Phenotype, Bin) %>%
    count() %>%
    mutate(Phenotype = case_match(Phenotype, "Ratio" ~ "Fraction", .default = Phenotype),
           Phenotypes = paste0(Marker, '-', Phenotype)) %>%
    ggplot(., aes(x = Phenotypes, y = n)) + 
        geom_bar(stat = "identity", aes(fill = Bin), position = "dodge") +
        scale_fill_viridis_d(direction = 1) + 
        labs(x = "Phenotype", y = "# cells") + 
        monocle3:::monocle_theme_opts() +
        theme(axis.title = element_text(color="black", size = 14), 
              axis.text.x = element_text(color="black", size = 11), 
              axis.text.y = element_text(color = "black", size = 11),
              axis.ticks = element_line(color = "black"))

In [ ]:
## Supplementary Figure 6B

options(repr.plot.width=9, repr.plot.height=4, repr.plot.res=300)
metadata %>%
    mutate(Phenotype = case_match(Phenotype, "Ratio" ~ "Fraction", .default = Phenotype),
           Phenotypes = paste0(Marker, '-', Phenotype)) %>%
    ggplot(., aes(x = Phenotypes, y = log10(RNA_UMIs))) + 
        geom_boxplot(aes(fill = Bin), lwd = 0.75, fatten = 0.75) +
        scale_fill_viridis_d(direction = 1) + 
        labs(x = "Phenotype", y = "log10 Total RNA UMIs") + 
        monocle3:::monocle_theme_opts() +
        theme(axis.title = element_text(color="black", size = 14), 
              axis.text.x = element_text(color="black", size = 11), 
              axis.text.y = element_text(color = "black", size = 11),
              axis.ticks = element_line(color = "black"))

In [ ]:
## Supplementary Figure 6C

options(repr.plot.width=9, repr.plot.height=4, repr.plot.res=300)
metadata %>%
    mutate(Phenotype = case_match(Phenotype, "Ratio" ~ "Fraction", .default = Phenotype),
           Phenotypes = paste0(Marker, '-', Phenotype)) %>%
    ggplot(., aes(x = Phenotypes, y = num_genes_expressed)) + 
        geom_boxplot(aes(fill = Bin), lwd = 0.75, fatten = 0.75, outlier.shape = NA) +
        scale_fill_viridis_d(direction = 1) + 
        labs(x = "Phenotype", y = "Number of genes expressed\n(UMI > 0)") + 
        ylim(c(NA, 1700)) + 
        monocle3:::monocle_theme_opts() +
        theme(axis.title = element_text(color="black", size = 14), 
              axis.text.x = element_text(color="black", size = 11), 
              axis.text.y = element_text(color = "black", size = 11),
              axis.ticks = element_line(color = "black"))

In [ ]:
pseudobulk_cds = function(cds_sub, sample_group = "Experiment", cell_group = "Embryo") {
    cf_ccs = new_cell_count_set(cds_sub,
                            sample_group = sample_group,
                            cell_group = cell_group)
    
    pb_cds = hooke:::pseudobulk_ccs_for_states(cf_ccs)
    
    return(pb_cds)
    
}

In [ ]:
pb_VCS = pseudobulk_cds(cds_VCS, sample_group = "Experiment", 
                                 cell_group = "Embryo")

In [ ]:
top_subset_genes = readRDS("../data/highly_dispersed_genes_3000_E15-E15.5.RDS")

In [ ]:
pb_metadata = as.data.frame(colData(pb_VCS)) %>%
                filter(num_cells_in_group >= 100) %>%
                mutate(sample_id = paste0(sample, "-", cell_group))
head(pb_metadata)

In [ ]:
embryo_ids = pb_metadata %>% arrange(cell_group) %>% pull(pseudobulk_id)

In [ ]:
corr_mat = matrix(0, nrow = length(embryo_ids), ncol = length(embryo_ids))

for (i in 1:length(embryo_ids)) {
    JAX_id = embryo_ids[i]
    for (j in 1:length(embryo_ids)) {
        VCS_id = embryo_ids[j] 
        pb_sub = pb_VCS[top_subset_genes, c(JAX_id, VCS_id)]
        expr_df = counts(pb_sub)
        corr_mat[i,j] = cor(log(expr_df[,1]+1), log(expr_df[,2]+1), method = "spearman")
    }
}

In [ ]:
## Supplementary Figure 6E

options(repr.plot.width=4, repr.plot.height=4, repr.plot.res=300)
expr_df %>%
    ggplot(., aes(x = `11_cell_group`, y = `12_cell_group`)) + 
        geom_point() + 
        geom_abline(slope = 1) +
        labs(x = "log (UMI + 1)\n\nE15.25 embryo 1", y = "E15.25 embryo 2\n\nlog (UMI + 1)") + 
        monocle3:::monocle_theme_opts() +
        theme(axis.title = element_text(color="black", size = 13), 
              axis.text.x = element_text(color="black", size = 11), 
              axis.text.y = element_text(color = "black", size = 11),
              axis.ticks = element_line(color = "black"))

In [ ]:
corr_df <- data.frame(corr = corr_mat[upper.tri(corr_mat)])
head(corr_df)

In [ ]:
## Supplementary Figure 6E

options(repr.plot.width=4, repr.plot.height=4, repr.plot.res=300)
ggplot(corr_df, aes(x = corr)) + 
    geom_histogram(n = 40, fill = "white", color = "black") + 
    labs(x = "Pairwise Spearman correlation", y= "Count") + 
    monocle3:::monocle_theme_opts() + 
    theme(axis.title = element_text(color="black", size = 13), 
          axis.text.x = element_text(color="black", size = 11), 
          axis.text.y = element_text(color = "black", size = 11),
          axis.ticks = element_line(color = "black"))

In [ ]:
my_colors = c("#C1A72F", "#FAD2D9", "#ED2891", "#F6B667", "#104A7F", "#9EDDF9", 
  "#3953A4", "#007EB5", "#B2509E", "#97D1A9", "#ED1C24", "#F8AFB3", 
  "#EA7075", "#754C29", "#D49DC7", "#CACCDB", "#D3C3E0", "#A084BD", 
  "#542C88", "#D97D25", "#6E7BA2", "#E8C51D", "#7E1918", "#DAF1FC", 
  "#00A99D", "#BBD642", "#00AEEF", "#BE1E2D", "#F9ED32", "#CEAC8F", 
  "#FBE3C7", "#F89420", "#009444", "#BCE7FA")

length(my_colors)

In [ ]:
plot_data = metadata %>%
    mutate(sample_id = paste0(Experiment, "-", Embryo)) %>%
    mutate(timepoint_num = as.numeric(stringr::str_remove(stringr::str_extract(sample_id, "E[0-9.]+"), "E")),
           sample_id = forcats::fct_reorder(sample_id, timepoint_num),
           sample_id = forcats::fct_relevel(sample_id, "1-E15.25-2", "2-E15.25-1", after = Inf),
           Phenotype = case_match(Phenotype, "Ratio" ~ "Fraction", .default = Phenotype),
           Phenotypes = paste0(Marker, '-', Phenotype)) %>%
    mutate(
        trajectory_refined = case_when(
          trajectory_refined == "Brain_and_spinal_cord" ~ "Neuroectoderm_and_glia",
          trajectory_refined == "Intermediate_progenitors" ~ "Intermediate_neuronal_progenitors",
          TRUE ~ trajectory_refined),
        trajectory_refined = gsub("_", " ", trajectory_refined))

counts_data <- plot_data %>% count(sample_id)

In [ ]:
set.seed(12)

options(repr.plot.width=6, repr.plot.height=5, repr.plot.res=300)
p1 <- ggplot(counts_data, aes(x = sample_id, y = n)) +
    geom_col(fill = "grey70", color = "black", linewidth = 0.2) +
    theme_minimal() +
    labs(y = "# cells", x = NULL) + 
    theme(
        axis.text.x = element_blank(), # Hide the text on the top plot so it doesn't overlap the bottom
        panel.grid.major.x = element_blank(),
        axis.title = element_text(color="black", size = 9), 
        axis.text.y = element_text(color = "black", size = 8))

p2 <- ggplot(plot_data, aes(x = sample_id, fill = trajectory_refined)) +
        geom_bar(position = "fill", color = "black", linewidth = 0.2) +
        labs(x = "", y = "% lineages", fill = "Lineage") + 
        scale_y_continuous(labels = scales::percent) +
        monocle3:::monocle_theme_opts() +
        guides(fill = guide_legend(ncol = 1, byrow = TRUE, keyheight = unit(0.25, "cm"))) +
        scale_fill_manual(values = my_colors[1:25]) + 
        theme(axis.title = element_text(color="black", size = 9), 
              axis.text.x = element_text(color="black", size = 8, angle = 45, hjust = 1), 
              axis.text.y = element_text(color = "black", size = 9),
              panel.grid.major.x = element_blank(), # Removes vertical grid lines for a cleaner look
              axis.ticks = element_line(color = "black"),
              legend.text = element_text(size = 7),
              legend.key.size = unit(0.4, "cm"),
              legend.spacing.y = unit(0.05, "cm"),
              legend.margin = margin(t = 0, r = 0, b = 0, l = 0, unit = "cm"))


my_legend <- get_legend(p2)

p2_bare <- p2 + theme(legend.position = "none")

stacked_plots <- plot_grid(
  p1, p2_bare, 
  ncol = 1, 
  align = "v",           
  rel_heights = c(1, 3)
)

final_plot <- plot_grid(
  stacked_plots, my_legend, 
  ncol = 2, 
  rel_heights = c(1, 0.4),
  rel_widths = c(1, 0.5)
)

## Supplementary Figure 6D
final_plot

## UMAP

In [ ]:
cds_combined = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered_umap2_final.RDS")

In [ ]:
metadata_combined = as.data.frame(colData(cds_combined))

In [ ]:
metadata_combined$trajectory_refined = NA
metadata_combined$trajectory_refined[metadata_combined$Cell %in% metadata$Cell] = metadata$trajectory_refined

In [ ]:
JAX_main_trajectory = metadata_combined[metadata_combined$Data == "JAX",]$Main_trajectory
JAX_main_cell_type = metadata_combined[metadata_combined$Data == "JAX",]$Main_cell_type

metadata_combined[metadata_combined$Data == "JAX",]$nn_majority_trajectory_pca = JAX_main_trajectory
metadata_combined[metadata_combined$Data == "JAX",]$nn_majority_trajectory_umap = JAX_main_trajectory
metadata_combined[metadata_combined$Data == "JAX",]$trajectory_refined = JAX_main_trajectory

metadata_combined[metadata_combined$Data == "JAX",]$nn_majority_cell_type_pca = JAX_main_cell_type
metadata_combined[metadata_combined$Data == "JAX",]$nn_majority_cell_type_umap = JAX_main_cell_type

cds_combined$nn_majority_trajectory_umap = metadata_combined$nn_majority_trajectory_umap
cds_combined$nn_majority_trajectory_pca = metadata_combined$nn_majority_trajectory_pca
cds_combined$nn_majority_cell_type_umap = metadata_combined$nn_majority_cell_type_umap
cds_combined$nn_majority_cell_type_pca = metadata_combined$nn_majority_cell_type_pca
cds_combined$trajectory_refined = metadata_combined$trajectory_refined

In [ ]:
subsample_cds = function(cds, n = 100000) {
    metadata = as.data.frame(colData(cds))
    
    metadata_JAX = metadata %>% filter(Data == "JAX") %>% sample_n(n)
    metadata_VCS = metadata %>% filter(Data == "VCS") %>% sample_n(n)

    cell_sampled = sample(c(metadata_JAX$Cell, metadata_VCS$Cell))
    
    cds_subsampled = cds[, cds$Cell %in% cell_sampled]
    
    return(cds_subsampled)
}


In [ ]:
cds_sampled = subsample_cds(cds_combined, 10000)

In [ ]:
cds_sampled

In [ ]:
## Supplementary Figure 6F
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=400)
g = plot_cells(cds_sampled, x = 1, y = 2,
           reduction_method = "UMAP", color_cells_by = "Data", 
           label_cell_groups = F,
           cell_size = 0.3) + 
        theme(legend.position = "none") + 
        scale_color_manual(values=c("#743282", "#f1a340"))

g$layers[[1]]$aes_params$alpha <- 0.2
g$layers[[2]]$aes_params$alpha <- NULL 
g + aes(alpha = Data) + scale_alpha_manual(values = c("JAX" = 0.5, "VCS" = 0.5))


In [ ]:
metadata_combined$UMAP_2d_1 = reducedDims(cds_combined)[['UMAP']][,1]
metadata_combined$UMAP_2d_2 = reducedDims(cds_combined)[['UMAP']][,2]
head(metadata_combined)

In [ ]:
res <- compute_lisi(metadata_combined[, c("UMAP_2d_1", "UMAP_2d_2")], 
                    metadata_combined, c("Data"))

In [ ]:
metadata_combined = metadata_combined %>%
    left_join(., res %>% rownames_to_column("Cell") %>% rename("lisi" = "Data"))

In [ ]:
metadata_combined %>%
    #filter(lisi < 1.1) %>%
    count(nn_majority_trajectory_pca)

In [ ]:
37315 / 70902
48602 / 349413

In [ ]:
metadata_combined %>%
    filter(lisi < 1.05) %>%
    count(nn_majority_trajectory_pca)

In [ ]:
## Supplementary Figure 6F

options(repr.plot.width=4, repr.plot.height=4, repr.plot.res=300)
ggplot(res, aes(x = Data)) + 
    geom_histogram(bins=100) + 
    labs(x = "Local Inverse Simpson's Index", y= "Count") + 
    monocle3:::monocle_theme_opts() + 
    theme(axis.title = element_text(color="black", size = 13), 
          axis.text.x = element_text(color="black", size = 11), 
          axis.text.y = element_text(color = "black", size = 11),
          axis.ticks = element_line(color = "black"))

In [ ]:
trajectory_comp = metadata_combined %>%
    group_by(Data, trajectory_refined) %>%
    summarise(n = n()) %>%
    filter(n >= 30) %>%
    group_by(trajectory_refined) %>%
    add_tally() %>% filter(nn == 2) %>% select(-nn) %>%
    group_by(Data) %>% mutate(sum_n = sum(n), prop_n = n / sum_n) %>%
    select(-c(n, sum_n)) %>%
    tidyr::pivot_wider(., names_from = "Data", 
                       values_from = "prop_n") 

In [ ]:
## Supplementary Figure 6G

options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=300)
trajectory_comp %>%
    ggplot(., aes(x = JAX, y = VCS)) +
        geom_point(size = 1.2, alpha = 0.5) +
        geom_abline(slope = 1) + 
        labs(x = "Reference lineage proportion", y= "VCS lineage proportion") + 
        monocle3:::monocle_theme_opts() + 
        theme(axis.title = element_text(color="black", size = 16), 
              axis.text.x = element_text(color="black", size = 12), 
              axis.text.y = element_text(color = "black", size = 12),
              axis.ticks = element_line(color = "black"))

In [ ]:
trajectory_comp = metadata_combined %>%
    group_by(Data, nn_majority_cell_type_pca) %>%
    summarise(n = n()) %>%
    filter(n >= 30) %>%
    group_by(nn_majority_cell_type_pca) %>%
    add_tally() %>% filter(nn == 2) %>% select(-nn) %>%
    group_by(Data) %>% mutate(sum_n = sum(n), prop_n = n / sum_n) %>%
    select(-c(n, sum_n)) %>%
    tidyr::pivot_wider(., names_from = "Data", 
                       values_from = "prop_n") 

In [ ]:
## Supplementary Figure 6G

options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=300)
trajectory_comp %>%
    ggplot(., aes(x = JAX, y = VCS)) +
        geom_point(size = 1.2, alpha = 0.5) +
        geom_abline(slope = 1) + 
        labs(x = "Reference cell type proportion", y= "VCS cell type proportion") + 
        monocle3:::monocle_theme_opts() + 
        theme(axis.title = element_text(color="black", size = 16), 
              axis.text.x = element_text(color="black", size = 12), 
              axis.text.y = element_text(color = "black", size = 12),
              axis.ticks = element_line(color = "black"))